# Notebook 07 – Machine Learning

## Objective

This notebook orchestrates the reusable machine learning framework developed for the IPL Analytics & Strategy Platform.

The notebook is responsible for:

- Validating the execution environment.
- Configuring machine learning settings.
- Executing the reusable ML pipeline.
- Displaying dataset information.
- Presenting training and evaluation metrics.
- Visualizing model performance.
- Summarizing execution results.

All machine learning logic resides inside the reusable `src` package. This notebook serves only as an orchestration layer, following the project's production-oriented architecture.

---

### Pipeline

Warehouse

↓
Analytics Dataset

↓
ML Pipeline

↓
Model Training

↓
Evaluation

↓
Results

In [1]:
from pathlib import Path
import warnings

import pandas as pd

from IPython.display import display

from config.config import (
    PROJECT_ROOT,
)

from src.workflows.ml_pipeline import run_ml_pipeline

warnings.filterwarnings("ignore")

In [2]:
print("=" * 80)
print("IPL Analytics & Strategy Platform")
print("Notebook 07 - Machine Learning")
print("=" * 80)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Working Dir  : {Path.cwd()}")

print("\nEnvironment validation completed successfully.")

IPL Analytics & Strategy Platform
Notebook 07 - Machine Learning
Project Root : D:\IPL-Analytics-Strategy-Platform
Working Dir  : D:\IPL-Analytics-Strategy-Platform\notebooks

Environment validation completed successfully.


In [3]:
TARGET_COLUMN = "winner"

MODELS = [
    "logistic_regression",
    "decision_tree",
    "random_forest",
    "xgboost",
    "lightgbm",
    "catboost"
]

print("Configuration")
print("-" * 40)
print(f"Target Column : {TARGET_COLUMN}")
print(f"Models        : {', '.join(MODELS)}")

Configuration
----------------------------------------
Target Column : winner
Models        : logistic_regression, decision_tree, random_forest, xgboost, lightgbm, catboost


In [4]:
print("=" * 80)
print("Running Machine Learning Pipelines")
print("=" * 80)

results = {}
for model in MODELS:
    print(f"Executing pipeline for model: {model}...")
    results[model] = run_ml_pipeline(
        target_column=TARGET_COLUMN,
        model_name=model,
    )

print("\nAll Machine Learning Pipelines completed successfully.")

Running Machine Learning Pipelines
Executing pipeline for model: logistic_regression...


Executing pipeline for model: decision_tree...
Executing pipeline for model: random_forest...


Executing pipeline for model: xgboost...


Executing pipeline for model: lightgbm...


Executing pipeline for model: catboost...



All Machine Learning Pipelines completed successfully.


In [5]:
print("=" * 80)
print("MODEL COMPARISON")
print("=" * 80)

comparison_data = []
from src.ml.constants import SUPPORTED_MODELS

for model_key, res in results.items():
    if res.status == "SUCCESS":
        eval_res = res.evaluation_result
        train_res = res.training_result
        comparison_data.append({
            "Model Key": model_key,
            "Model Name": SUPPORTED_MODELS[model_key],
            "CV Acc (Mean)": eval_res.cv_mean_accuracy,
            "CV Acc (Std)": eval_res.cv_std_accuracy,
            "Test Accuracy": eval_res.accuracy,
            "Precision": eval_res.precision,
            "Recall": eval_res.recall,
            "F1 Score": eval_res.f1_score,
            "Training Time (s)": train_res.training_time,
            "Pipeline Time (s)": res.execution_time,
        })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True))

# Choose best model
best_row = comparison_df.loc[comparison_df["Test Accuracy"].idxmax()]
best_model_key = best_row["Model Key"]
print(f"\nBest Model: {best_row['Model Name']} with CV Accuracy: {best_row['CV Acc (Mean)']:.4f} and Test Accuracy: {best_row['Test Accuracy']:.4f}")

MODEL COMPARISON


,Model Key,Model Name,CV Acc (Mean),CV Acc (Std),Test Accuracy,Precision,Recall,F1 Score,Training Time (s),Pipeline Time (s)
0,catboost,CatBoost,0.508253,0.020498,0.532787,0.523496,0.532787,0.519673,5.094731,25.255636
1,xgboost,XGBoost,0.502067,0.011066,0.512295,0.507431,0.512295,0.505562,0.689732,2.982832
2,random_forest,Random Forest,0.451763,0.013566,0.487705,0.477368,0.487705,0.475544,0.426357,2.662821
3,lightgbm,LightGBM,0.518536,0.039035,0.487705,0.476587,0.487705,0.479701,2.370889,4.990867
4,decision_tree,Decision Tree,0.391234,0.031751,0.409836,0.411773,0.409836,0.406544,0.005874,0.170217
5,logistic_regression,Logistic Regression,0.249500,0.012669,0.270492,0.237778,0.270492,0.247243,0.656291,3.688167



Best Model: CatBoost with CV Accuracy: 0.5083 and Test Accuracy: 0.5328


In [6]:
pipeline_results = results[best_model_key]
training_result = pipeline_results.training_result
evaluation_result = pipeline_results.evaluation_result
preprocessed_dataset = pipeline_results.preprocessed_dataset

# Save ML artifacts to models/ directory
from config.config import MODELS_DIR

# 1. Save benchmark results
comparison_df.to_csv(MODELS_DIR / "benchmark_results.csv", index=False)

# 2. Save classification report as JSON
evaluation_result.classification_report.to_json(
    MODELS_DIR / "classification_report.json",
    indent=4,
)

# 3. Save confusion matrix
evaluation_result.confusion_matrix.to_csv(
    MODELS_DIR / "confusion_matrix.csv",
    index=True,
)

# 4. Save feature importance
if evaluation_result.feature_importance is not None:
    evaluation_result.feature_importance.to_csv(
        MODELS_DIR / "feature_importance.csv",
        index=False,
    )
else:
    pd.DataFrame(columns=["feature", "importance"]).to_csv(
        MODELS_DIR / "feature_importance.csv",
        index=False,
    )

In [7]:
print("=" * 80)
print("Dataset Summary")
print("=" * 80)

dataset_summary = pd.DataFrame(
    {
        "Metric": [
            "Training Samples",
            "Testing Samples",
            "Feature Count",
            "Target Column",
        ],
        "Value": [
            len(preprocessed_dataset.X_train),
            len(preprocessed_dataset.X_test),
            preprocessed_dataset.X_train.shape[1],
            TARGET_COLUMN,
        ],
    }
)

display(dataset_summary)

Dataset Summary


,Metric,Value
0,Training Samples,974
1,Testing Samples,244
2,Feature Count,8
3,Target Column,winner


In [8]:
print("=" * 80)
print("Training Summary")
print("=" * 80)

training_summary = pd.DataFrame(
    {
        "Metric": [
            "Model",
            "Estimator",
        ],
        "Value": [
            training_result.model_name,
            type(training_result.model).__name__,
        ],
    }
)

display(training_summary)

Training Summary


,Metric,Value
0,Model,catboost
1,Estimator,CatBoostClassifier


In [9]:
print("=" * 80)
print("Evaluation Metrics")
print("=" * 80)

metrics_df = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
        ],
        "Score": [
            evaluation_result.accuracy,
            evaluation_result.precision,
            evaluation_result.recall,
            evaluation_result.f1_score,
        ],
    }
)

display(metrics_df)

Evaluation Metrics


,Metric,Score
0,Accuracy,0.532787
1,Precision,0.523496
2,Recall,0.532787
3,F1 Score,0.519673


In [10]:
print("=" * 80)
print("Feature Importance")
print("=" * 80)

if evaluation_result.feature_importance is not None:

    feature_importance_df = (
        evaluation_result.feature_importance
        .sort_values(
            by="importance",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    display(feature_importance_df)

else:
    print("Feature importance is not available for this model.")

Feature Importance


,feature,importance
0,team2,23.333159
1,team1,20.662129
2,season,20.422527
3,toss_winner,11.661389
4,venue,9.839973
5,city,9.343216
6,toss_decision,4.737608
7,match_type,0.000000


In [11]:
print("=" * 80)
print("Confusion Matrix")
print("=" * 80)

confusion_matrix_df = pd.DataFrame(
    evaluation_result.confusion_matrix
)

display(confusion_matrix_df)

Confusion Matrix


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,24,0,0,0,0,1,0,0,0,0,3,0,0,1,0,1,0
1,0,2,0,0,0,0,1,0,0,0,1,0,0,1,0,1,0
2,1,0,3,0,0,0,0,0,2,1,3,0,1,1,0,0,0
3,1,2,0,4,0,0,1,0,0,0,1,0,0,0,0,1,3
4,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
5,1,0,1,0,0,4,0,0,1,0,1,0,0,1,0,0,0
6,0,0,0,0,1,0,5,0,4,0,0,0,0,2,0,4,1
7,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8,1,0,2,0,0,0,0,0,15,1,1,0,2,3,0,1,2
9,1,0,0,0,0,0,0,0,2,2,0,0,0,1,0,1,0


In [12]:
print("=" * 80)
print("Classification Report")
print("=" * 80)

display(
    evaluation_result.classification_report
)

Classification Report


,precision,recall,f1-score,support
Chennai Super Kings,0.615385,0.800000,0.695652,30.000000
Deccan Chargers,0.500000,0.333333,0.400000,6.000000
Delhi Capitals,0.333333,0.250000,0.285714,12.000000
Delhi Daredevils,0.571429,0.307692,0.400000,13.000000
Gujarat Lions,0.500000,0.333333,0.400000,3.000000
Gujarat Titans,0.666667,0.444444,0.533333,9.000000
Kings XI Punjab,0.500000,0.294118,0.370370,17.000000
Kochi Tuskers Kerala,0.000000,0.000000,0.000000,1.000000
Kolkata Knight Riders,0.535714,0.535714,0.535714,28.000000
Lucknow Super Giants,0.285714,0.285714,0.285714,7.000000


In [13]:
print("=" * 80)
print("Execution Summary")
print("=" * 80)

execution_summary = pd.DataFrame(
    {
        "Metric": [
            "Pipeline Status",
            "Model",
            "Target",
            "Execution Time (Seconds)",
            "Training Samples",
            "Testing Samples",
            "Accuracy",
        ],
        "Value": [
            pipeline_results.status,
            training_result.model_name,
            TARGET_COLUMN,
            round(
                pipeline_results.execution_time,
                2,
            ),
            len(preprocessed_dataset.X_train),
            len(preprocessed_dataset.X_test),
            round(
                evaluation_result.accuracy,
                4,
            ),
        ],
    }
)

display(execution_summary)

print("\nNotebook completed successfully.")


Execution Summary


,Metric,Value
0,Pipeline Status,SUCCESS
1,Model,catboost
2,Target,winner
3,Execution Time (Seconds),25.26
4,Training Samples,974
5,Testing Samples,244
6,Accuracy,0.5328



Notebook completed successfully.
